In [28]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [29]:
!pip install -q transformers accelerate bitsandbytes

In [30]:
from kaggle_secrets import UserSecretsClient
import wandb
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()
print("W&B logged in!")

W&B logged in!


In [31]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# ── Load Data ──────────────────────────────────────────────────
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ["A", "B", "C", "D", "E"]

def mapk(actual, predicted, k=3):
    def apk(a, p):
        score, hits = 0.0, 0
        for i, pi in enumerate(p[:k]):
            if pi == a:
                hits += 1
                score += hits / (i + 1)
        return score
    return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# ── Build pairwise ranking features ───────────────────────────
# Key insight: instead of "is this option correct?" (binary)
# we ask "is option X better than option Y?" (ranking)
# This is what your friend did — rank-based not classify-based

def build_texts(df):
    """
    For each question, create 5 texts:
    prompt + option (word ngrams catch semantics,
    char ngrams catch subtle spelling/word differences)
    """
    texts = []
    for _, row in df.iterrows():
        for opt in OPTION_COLS:
            # Full text combination
            combined = (
                str(row["prompt"]) + " " +
                str(row["prompt"]) +  # repeat prompt to give it more weight
                " [SEP] " +
                str(row[opt])
            )
            texts.append(combined)
    return texts

train_texts = build_texts(train_df)
test_texts  = build_texts(test_df)

# ── Labels: rank-based (not binary 0/1) ───────────────────────
# Correct option gets score 4, others get 0,1,2,3 randomly
# This teaches the model to RANK not just classify
label2idx = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_labels = []
for _, row in train_df.iterrows():
    correct_idx = label2idx[row["answer"]]
    for i in range(5):
        # Correct = 4 (highest rank), wrong = 0
        train_labels.append(4 if i == correct_idx else 0)

# ── TF-IDF: BOTH word and character ngrams ─────────────────────
# Word ngrams: catch semantic meaning
# Char ngrams: catch subtle differences in similar paragraphs
# This is the KEY difference from basic TF-IDF

from sklearn.pipeline import Pipeline, FeatureUnion

word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),      # unigrams, bigrams, trigrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",       # character ngrams within word boundaries
    ngram_range=(3, 5),       # 3,4,5 char ngrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

combined_features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

# ── Train/Val Split ────────────────────────────────────────────
n = len(train_df)
idx = np.arange(n)
tr_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

tr_text_idx  = [i*5+j for i in tr_idx  for j in range(5)]
val_text_idx = [i*5+j for i in val_idx for j in range(5)]

tr_texts_split   = [train_texts[i] for i in tr_text_idx]
val_texts_split  = [train_texts[i] for i in val_text_idx]
tr_labels_split  = [train_labels[i] for i in tr_text_idx]

print("Fitting TF-IDF features (word + char ngrams)...")
X_train = combined_features.fit_transform(tr_texts_split)
X_val   = combined_features.transform(val_texts_split)
X_test  = combined_features.transform(test_texts)
print(f"Feature matrix shape: {X_train.shape}")

# ── Logistic Regression ────────────────────────────────────────
print("Training Logistic Regression...")
clf = LogisticRegression(
    C=5.0,
    max_iter=2000,
    solver="saga",
    n_jobs=-1
)
clf.fit(X_train, tr_labels_split)
print("Done!")

# ── Rank-based Prediction ──────────────────────────────────────
# Use predict_proba score for class 4 (correct option)
# This gives a continuous score per option → rank them

def get_scores_and_predict(X, n_questions):
    # Get probability of being the correct answer (class 4)
    probs = clf.predict_proba(X)
    # Find which column corresponds to class 4
    class4_idx = list(clf.classes_).index(4)
    scores = probs[:, class4_idx]
    # Reshape to (n_questions, 5)
    scores_matrix = scores.reshape(n_questions, 5)
    # Rank: top 3 per question
    top3 = np.argsort(-scores_matrix, axis=1)[:, :3]
    return [[OPTION_COLS[i] for i in row] for row in top3]

# ── Validate ───────────────────────────────────────────────────
val_preds  = get_scores_and_predict(X_val, len(val_idx))
true_labels = [train_df.iloc[i]["answer"] for i in val_idx]
val_map3   = mapk(true_labels, val_preds)
print(f"\nValidation MAP@3: {val_map3:.4f}")

# ── Test Submission ────────────────────────────────────────────
test_preds = get_scores_and_predict(X_test, len(test_df))

submission = pd.DataFrame({
    "id":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds]
})



submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head(10))

Fitting TF-IDF features (word + char ngrams)...
Feature matrix shape: (8000, 53456)
Training Logistic Regression...
Done!

Validation MAP@3: 0.9962
Saved submission.csv
   id Prediction
0   1      A E D
1   2      B E C
2   3      B E D
3   4      E C D
4   5      C A D
5   6      D A C
6   7      E D C
7   8      B E A
8   9      C D E
9  10      B D C


In [32]:
print(submission.head(20))

    id Prediction
0    1      A E D
1    2      B E C
2    3      B E D
3    4      E C D
4    5      C A D
5    6      D A C
6    7      E D C
7    8      B E A
8    9      C D E
9   10      B D C
10  11      A C B
11  12      D E B
12  13      C A E
13  14      C E D
14  15      E D C
15  16      A C E
16  17      E D B
17  18      B E D
18  19      A E B
19  20      D B C


In [33]:
# Initialize W&B run
wandb.init(
    project="22f3001070-t22026",   
    name="tfidf-logreg-word-char-ngram"
)

# Log everything
wandb.log({
    "model": "TF-IDF + Logistic Regression",
    "word_ngram_range": "(1,3)",
    "char_ngram_range": "(3,5)",
    "max_features_word": 200000,
    "max_features_char": 200000,
    "C": 5.0,
    "solver": "saga",
    "label_type": "rank-based (0 vs 4)",
    "train_size": len(tr_idx) * 5,
    "val_size": len(val_idx) * 5,
    "val_map3": val_map3,
})

print(f"Logged to W&B — MAP@3: {val_map3:.4f}")
wandb.finish()

Logged to W&B — MAP@3: 0.9962


C,▁
max_features_char,▁
max_features_word,▁
train_size,▁
val_map3,▁
val_size,▁
C,5
char_ngram_range,"(3,5)"
label_type,rank-based (0 vs 4)
max_features_char,200000
max_features_word,200000


In [34]:
# ##MILESTONE - 1

# import pandas as pd
# import numpy as np
# import string
# from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
# from sklearn.metrics.pairwise import cosine_similarity

# # Load the data
# train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # ============================================
# # Question 1: Frequency distribution of correct answers
# # ============================================
# print("=" * 50)
# print("QUESTION 1: Frequency Distribution of Correct Answers")
# print("=" * 50)

# answer_freq = train_df['answer'].value_counts().sort_index()
# print("Frequency of each answer:")
# print(answer_freq)

# most_frequent = answer_freq.max()
# least_frequent = answer_freq.min()
# sum_most_least = most_frequent + least_frequent

# print(f"\nMost frequent option count: {most_frequent}")
# print(f"Least frequent option count: {least_frequent}")
# print(f"Sum of most and least frequent: {sum_most_least}")

# # ============================================
# # Question 2: Vocabulary size after cleaning prompts
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 2: Vocabulary Size After Cleaning")
# print("=" * 50)

# def clean_text(text):
#     # Convert to lowercase
#     text = text.lower()
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     return text

# # Clean all prompts
# train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# # Get all unique words
# all_words = set()
# for prompt in train_df['cleaned_prompt']:
#     words = prompt.split()
#     all_words.update(words)

# vocab_size = len(all_words)
# print(f"Total unique words (vocabulary size): {vocab_size}")

# # ============================================
# # Question 3: Words left in Row ID 1 after removing stop words
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 3: Words in Row ID 1 After Removing Stop Words")
# print("=" * 50)

# # Get cleaned prompt for Row ID 1
# row1_prompt = train_df[train_df['id'] == 1]['cleaned_prompt'].values[0]

# # Split into words
# row1_words = row1_prompt.split()

# # Filter out stop words
# row1_filtered = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

# words_left = len(row1_filtered)
# print(f"Row ID 1 cleaned prompt (first 100 chars): {row1_prompt[:100]}...")
# print(f"Words left after removing stop words: {words_left}")

# # ============================================
# # Question 4: TF-IDF Vectorizer vocabulary size
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 4: TF-IDF Vectorizer Vocabulary Size")
# print("=" * 50)

# # Combine prompt and all options into single documents for each row
# combined_texts = []
# for idx, row in train_df.iterrows():
#     # Combine prompt with all options
#     combined = row['prompt'] + ' ' + row['A'] + ' ' + row['B'] + ' ' + row['C'] + ' ' + row['D'] + ' ' + row['E']
#     combined_texts.append(combined)

# # Fit TF-IDF vectorizer
# tfidf_vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_vectorizer.fit(combined_texts)

# feature_columns = len(tfidf_vectorizer.get_feature_names_out())
# print(f"Number of feature columns (vocabulary size): {feature_columns}")

# # ============================================
# # Question 5: Cosine similarity between prompt and option A for Row ID 1
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)")
# print("=" * 50)

# # Get Row ID 1 data
# row1 = train_df[train_df['id'] == 1].iloc[0]

# # Transform prompt and option A separately
# prompt_vector = tfidf_vectorizer.transform([row1['prompt']])
# option_a_vector = tfidf_vectorizer.transform([row1['A']])

# # Calculate cosine similarity
# similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

# print(f"Prompt: {row1['prompt'][:100]}...")
# print(f"Option A: {row1['A'][:100]}...")
# print(f"Cosine similarity score: {similarity_score:.4f}")

# # ============================================
# # Question 6: Percentage where highest similarity matches correct answer
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 6: Percentage of Highest Similarity Matching Correct Answer")
# print("=" * 50)

# correct_matches = 0
# total_rows = len(train_df)

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Vectorize each option and calculate similarity
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Find option with highest similarity
#     highest_sim_option = max(similarities, key=similarities.get)
    
#     # Check if matches correct answer
#     if highest_sim_option == row['answer']:
#         correct_matches += 1

# percentage = (correct_matches / total_rows) * 100
# print(f"Rows where highest similarity matches correct answer: {correct_matches}/{total_rows}")
# print(f"Percentage: {percentage:.2f}%")

# # ============================================
# # Question 7: MAP@3 score for prediction C A B when answer is C
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 7: MAP@3 for prediction C A B (answer is C)")
# print("=" * 50)

# def calculate_map_at_3(ground_truth, predictions):
#     """
#     Calculate MAP@3 for a single question
#     predictions: list of 3 predicted answers in order
#     """
#     for i, pred in enumerate(predictions):
#         if pred == ground_truth:
#             return 1.0 / (i + 1)  # 1/k where k is the position (1-indexed)
#     return 0.0  # Not in top 3

# # Example: answer is C, prediction is C A B
# map_score_q7 = calculate_map_at_3('C', ['C', 'A', 'B'])
# print(f"Ground truth: C, Prediction: C A B")
# print(f"MAP@3 score: {map_score_q7}")

# # ============================================
# # Question 8: MAP@3 score for prediction D B E when answer is B
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 8: MAP@3 for prediction D B E (answer is B)")
# print("=" * 50)

# map_score_q8 = calculate_map_at_3('B', ['D', 'B', 'E'])
# print(f"Ground truth: B, Prediction: D B E")
# print(f"MAP@3 score: {map_score_q8}")

# # ============================================
# # Question 9: Majority Class Baseline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 9: Majority Class Baseline MAP@3")
# print("=" * 50)

# # Get frequency of answers
# answer_counts = train_df['answer'].value_counts()
# print("Answer frequencies:")
# print(answer_counts)

# # Get top 3 most frequent answers
# top3_answers = answer_counts.head(3).index.tolist()
# print(f"Top 3 most frequent answers: {top3_answers}")

# # Calculate MAP@3 for majority baseline
# majority_scores = []
# for idx, row in train_df.iterrows():
#     ground_truth = row['answer']
#     predictions = top3_answers  # Always predict the same top 3
#     score = calculate_map_at_3(ground_truth, predictions)
#     majority_scores.append(score)

# overall_majority_map = np.mean(majority_scores)
# print(f"Overall MAP@3 for Majority Class Baseline: {overall_majority_map:.4f}")

# # ============================================
# # Question 10: TF-IDF Pipeline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 10: TF-IDF Pipeline MAP@3")
# print("=" * 50)

# tfidf_scores = []

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Calculate similarity for each option
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Sort options by similarity (highest to lowest)
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
#     # Get top 3 predictions
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     # Calculate MAP@3 for this row
#     ground_truth = row['answer']
#     score = calculate_map_at_3(ground_truth, top3_predictions)
#     tfidf_scores.append(score)

# overall_tfidf_map = np.mean(tfidf_scores)
# print(f"Overall MAP@3 for TF-IDF Pipeline: {overall_tfidf_map:.4f}")

# # ============================================
# # Create sample submission file
# # ============================================
# print("\n" + "=" * 50)
# print("Creating Sample Submission File")
# print("=" * 50)

# # Create predictions for test set using TF-IDF approach
# submission_predictions = []

# for idx, row in test_df.iterrows():
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     submission_predictions.append({
#         'ID': row['id'],
#         'Prediction': ' '.join(top3_predictions)
#     })

# # Create submission DataFrame
# submission_df = pd.DataFrame(submission_predictions)

# # Save to CSV
# submission_df.to_csv('sample_submission.csv', index=False)

# print(f"Sample submission file created with {len(submission_df)} predictions")
# print("\nFirst few predictions:")
# print(submission_df.head())

In [35]:
# # MILESTONE 2 
# !pip install -q transformers datasets sentence-transformers

# import torch
# import numpy as np
# import pandas as pd
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModel, pipeline as hf_pipeline
# from sentence_transformers import SentenceTransformer, util
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # ── Load Data ────────────────────────────────────────────────
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# dataset  = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
# OPTION_COLS = ["A", "B", "C", "D", "E"]

# def mapk(actual, predicted, k=3):
#     def apk(a, p):
#         score, hits = 0.0, 0
#         for i, pi in enumerate(p[:k]):
#             if pi == a:
#                 hits += 1
#                 score += hits / (i + 1)
#         return score
#     return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# # ── Q1: combined_text length at index 51 ─────────────────────
# dataset = dataset.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
# q1 = len(dataset[51]["combined_text"])
# print(f"Q1 - combined_text length at index 51: {q1}")

# # ── Q2: BERT vocab size ───────────────────────────────────────
# bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# q2 = bert_tokenizer.vocab_size
# print(f"Q2 - BERT vocab size: {q2}")

# # ── Q3: SEP token ID ─────────────────────────────────────────
# q3 = bert_tokenizer.sep_token_id
# print(f"Q3 - [SEP] token ID: {q3}")

# # ── Q4: Shape of input_ids tensor ────────────────────────────
# # ── Q4: Shape of input_ids tensor ────────────────────────────
# encoded = bert_tokenizer(
#     list(dataset["prompt"]),   # ← add list() here
#     padding="max_length",
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )
# q4 = tuple(encoded["input_ids"].shape)
# print(f"Q4 - input_ids shape: {q4}")

# # ── Q5: Attention head dimension ─────────────────────────────
# q5 = 768 // 12
# print(f"Q5 - Each attention head dimension: {q5}")

# # ── Q6: last_hidden_state shape for row 0 ────────────────────
# bert_model = AutoModel.from_pretrained("bert-base-uncased")
# bert_model.eval()
# inputs_row0 = bert_tokenizer(dataset[0]["prompt"], return_tensors="pt")
# with torch.no_grad():
#     outputs_row0 = bert_model(**inputs_row0)
# q6 = tuple(outputs_row0.last_hidden_state.shape)
# print(f"Q6 - last_hidden_state shape: {q6}")

# # ── Q7: Sum of first 5 CLS values ────────────────────────────
# cls_emb = outputs_row0.last_hidden_state[0, 0, :]
# q7 = round(cls_emb[:5].sum().item(), 4)
# print(f"Q7 - Sum of first 5 CLS values: {q7}")

# # ── Q8: Attention CLS→fusion in last layer, head 0 ───────────
# bert_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
# bert_attn.eval()
# text = "Light-ion fusion is a technique."
# inputs_attn = bert_tokenizer(text, return_tensors="pt")
# with torch.no_grad():
#     outputs_attn = bert_attn(**inputs_attn)
# tokens = bert_tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])
# print(f"Tokens: {tokens}")
# fusion_idx = tokens.index("fusion")
# q8 = round(outputs_attn.attentions[-1][0, 0, 0, fusion_idx].item(), 4)
# print(f"Q8 - Attention CLS→fusion: {q8}")

# # ── Q9: Cosine sim prompt vs Option B row 0 ──────────────────
# st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# emb_prompt = st_model.encode(dataset[0]["prompt"], convert_to_tensor=True)
# emb_b      = st_model.encode(dataset[0]["B"],      convert_to_tensor=True)
# q9 = round(util.cos_sim(emb_prompt, emb_b).item(), 4)
# print(f"Q9 - Cosine similarity prompt vs B: {q9}")

# # ── Q10: MiniLM MAP@3 + count where MiniLM beats TF-IDF ──────
# def tfidf_predict(df):
#     preds = []
#     all_texts = df["prompt"].tolist() + df[OPTION_COLS].values.flatten().tolist()
#     vec = TfidfVectorizer(ngram_range=(1,2), max_features=50000)
#     vec.fit(all_texts)
#     for _, row in df.iterrows():
#         pv = vec.transform([row["prompt"]])
#         ov = vec.transform([row[c] for c in OPTION_COLS])
#         sims = cosine_similarity(pv, ov)[0]
#         top3 = [OPTION_COLS[i] for i in np.argsort(sims)[::-1][:3]]
#         preds.append(top3)
#     return preds

# def minilm_predict(df):
#     preds = []
#     for _, row in df.iterrows():
#         pemb  = st_model.encode(row["prompt"], convert_to_tensor=True)
#         oembs = st_model.encode([row[c] for c in OPTION_COLS], convert_to_tensor=True)
#         sims  = util.cos_sim(pemb, oembs)[0].cpu().numpy()
#         top3  = [OPTION_COLS[i] for i in np.argsort(sims)[::-1][:3]]
#         preds.append(top3)
#     return preds

# print("Running TF-IDF pipeline...")
# tfidf_preds  = tfidf_predict(train_df)
# print("Running MiniLM pipeline...")
# minilm_preds = minilm_predict(train_df)

# true_labels  = train_df["answer"].tolist()
# q10a = round(mapk(true_labels, minilm_preds), 4)
# print(f"Q10a - MiniLM MAP@3: {q10a}")

# q10b = sum(
#     1 for true, tp, mp in zip(true_labels, tfidf_preds, minilm_preds)
#     if true not in tp and true in mp
# )
# print(f"Q10b - Count MiniLM beats TF-IDF: {q10b}")

# # ── Q11: Zero-shot classification softmax ────────────────────
# zsc = hf_pipeline("zero-shot-classification")
# row1 = dataset[1]
# candidate_labels = [row1["A"], row1["B"], row1["C"]]
# result_softmax = zsc(row1["prompt"], candidate_labels)
# q11 = round(result_softmax["scores"][0], 4)
# print(f"Q11 - Top ranked score (softmax): {q11}")

# # ── Q12: Multi-label sigmoid vs softmax difference ───────────
# result_sigmoid = zsc(row1["prompt"], candidate_labels, multi_label=True)
# softmax_sum = sum(result_softmax["scores"])
# sigmoid_sum = sum(result_sigmoid["scores"])
# q12 = round(abs(softmax_sum - sigmoid_sum), 4)
# print(f"Q12 - Softmax sum: {round(softmax_sum,4)} | Sigmoid sum: {round(sigmoid_sum,4)}")
# print(f"Q12 - Absolute difference: {q12}")

# # ── Q13: Flan-T5 generative output ───────────────────────────
# from transformers import T5ForConditionalGeneration, T5Tokenizer

# flan_tok   = T5Tokenizer.from_pretrained("google/flan-t5-small")
# flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
# flan_model.eval()

# row0 = dataset[0]
# input_str = (
#     f"Question: {row0['prompt']}. "
#     f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
#     f"Answer with just the letter A or B."
# )

# inputs = flan_tok(input_str, return_tensors="pt")
# with torch.no_grad():
#     out = flan_model.generate(**inputs, max_new_tokens=5)
# q13 = flan_tok.decode(out[0], skip_special_tokens=True)
# print(f"Q13 - Flan-T5 output: '{q13}'")
# # ── FINAL SUMMARY ─────────────────────────────────────────────
# print("\n========== ALL ANSWERS ==========")
# print(f"Q1  : {q1}")
# print(f"Q2  : {q2}")
# print(f"Q3  : {q3}")
# print(f"Q4  : {q4}")
# print(f"Q5  : {q5}")
# print(f"Q6  : {q6}")
# print(f"Q7  : {q7}")
# print(f"Q8  : {q8}")
# print(f"Q9  : {q9}")
# print(f"Q10a: {q10a}")
# print(f"Q10b: {q10b}")
# print(f"Q11 : {q11}")
# print(f"Q12 : {q12}")
# print(f"Q13 : {q13}")
# print("=================================")